In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:

# # Conditional GAN (cGAN) vs Vanilla GAN on Fashion-MNIST

import os
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid


SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cpu")
print(f"Running on : {device}")
print("Note: CUDA P100 (sm_60) is incompatible with this PyTorch build → CPU mode.")

LATENT_DIM        = 100
NUM_CLASSES       = 10
IMG_SIZE          = 28
CHANNELS          = 1
IMG_DIM           = IMG_SIZE * IMG_SIZE * CHANNELS   # 784

BATCH_SIZE        = 256    
LR                = 0.0002
BETA1             = 0.5
BETA2             = 0.999
NUM_EPOCHS        = 50    
SAMPLE_EVERY      = 10
SAMPLES_PER_CLASS = 10

CLASS_NAMES = [
    "T-shirt/top", "Trouser",  "Pullover", "Dress",  "Coat",
    "Sandal",      "Shirt",    "Sneaker",  "Bag",    "Ankle boot"
]

print(f"Batch size : {BATCH_SIZE}")
print(f"Epochs     : {NUM_EPOCHS}")
print(f"Device     : {device}")


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False
)

print(f"Dataset size : {len(train_dataset):,}")
print(f"Batches/epoch: {len(train_loader)}")


fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Fashion-MNIST – Sample Images", fontsize=14, fontweight='bold')
shown, idx = set(), 0
while len(shown) < NUM_CLASSES and idx < len(train_dataset):
    img, label = train_dataset[idx]
    if label not in shown:
        r, c = label // 5, label % 5
        axes[r, c].imshow(img.squeeze(), cmap='gray')
        axes[r, c].set_title(CLASS_NAMES[label], fontsize=9)
        axes[r, c].axis('off')
        shown.add(label)
    idx += 1
plt.tight_layout()
plt.savefig("sample_images.png", dpi=120)
plt.show()
print("Saved: sample_images.png")


class VanillaGenerator(nn.Module):
    def __init__(self, latent_dim, img_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(256, momentum=0.8),

            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(512, momentum=0.8),

            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(1024, momentum=0.8),

            nn.Linear(1024, img_dim),
            nn.Tanh()
        )
    def forward(self, z):
        return self.net(z)


class VanillaDiscriminator(nn.Module):
    def __init__(self, img_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, img):
        return self.net(img)


class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim, num_classes, img_dim, embed_dim=50):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + embed_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(256, momentum=0.8),

            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(512, momentum=0.8),

            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(1024, momentum=0.8),

            nn.Linear(1024, img_dim),
            nn.Tanh()
        )
    def forward(self, z, labels):
        return self.net(torch.cat([z, self.label_emb(labels)], dim=1))


class ConditionalDiscriminator(nn.Module):
    def __init__(self, img_dim, num_classes, embed_dim=50):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(img_dim + embed_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, img, labels):
        return self.net(torch.cat([img, self.label_emb(labels)], dim=1))

def weights_init_cpu(m):
    """
    CPU-safe weight initialisation.
    Uses torch.no_grad() + normal_ / constant_ directly (no CUDA kernel calls).
    """
    with torch.no_grad():
        classname = m.__class__.__name__
        if classname == 'Linear':
            
            fan_in  = m.weight.size(1)
            fan_out = m.weight.size(0)
            std = (2.0 / (fan_in + fan_out)) ** 0.5
            m.weight.normal_(0.0, std)
            if m.bias is not None:
                m.bias.fill_(0.0)
        elif classname == 'BatchNorm1d':
            m.weight.fill_(1.0)
            m.bias.fill_(0.0)
        elif classname == 'Embedding':
            m.weight.normal_(0.0, 0.02)


van_G = VanillaGenerator(LATENT_DIM, IMG_DIM).to(device)
van_D = VanillaDiscriminator(IMG_DIM).to(device)
van_G.apply(weights_init_cpu)
van_D.apply(weights_init_cpu)


c_G = ConditionalGenerator(LATENT_DIM, NUM_CLASSES, IMG_DIM).to(device)
c_D = ConditionalDiscriminator(IMG_DIM, NUM_CLASSES).to(device)
c_G.apply(weights_init_cpu)
c_D.apply(weights_init_cpu)

opt_van_G = optim.Adam(van_G.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_van_D = optim.Adam(van_D.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_c_G   = optim.Adam(c_G.parameters(),   lr=LR, betas=(BETA1, BETA2))
opt_c_D   = optim.Adam(c_D.parameters(),   lr=LR, betas=(BETA1, BETA2))

criterion = nn.BCELoss()

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Vanilla G  params: {count_params(van_G):,}")
print(f"Vanilla D  params: {count_params(van_D):,}")
print(f"cGAN    G  params: {count_params(c_G):,}")
print(f"cGAN    D  params: {count_params(c_D):,}")


fixed_noise  = torch.randn(NUM_CLASSES * SAMPLES_PER_CLASS, LATENT_DIM)
fixed_labels = torch.arange(NUM_CLASSES).repeat_interleave(SAMPLES_PER_CLASS)


def save_image_grid(images, epoch, model_name, nrow=SAMPLES_PER_CLASS, title=""):
    images = ((images + 1) / 2).clamp(0, 1)
    grid   = make_grid(images, nrow=nrow, padding=2, normalize=False)
    npgrid = grid.permute(1, 2, 0).cpu().numpy()

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(npgrid, cmap='gray', aspect='auto')
    ax.axis('off')
    ax.set_title(f"{title} – Epoch {epoch}", fontsize=14, fontweight='bold')

    h = npgrid.shape[0] / NUM_CLASSES
    for i, name in enumerate(CLASS_NAMES):
        ax.text(-5, (i + 0.5) * h, name, va='center', ha='right',
                fontsize=7.5, color='black')

    fname = f"{model_name}_epoch_{epoch:03d}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    return fname


def plot_losses(g_losses, d_losses, title, filename):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    epochs = range(1, len(g_losses) + 1)

    ax1.plot(epochs, g_losses, label='Generator',     color='#E74C3C', linewidth=1.5)
    ax1.plot(epochs, d_losses, label='Discriminator', color='#3498DB', linewidth=1.5)
    ax1.set_title(f"{title} – Training Losses")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    w = 5
    def smooth(arr):
        return np.convolve(arr, np.ones(w)/w, mode='valid')

    if len(g_losses) > w:
        se = range(w, len(g_losses) + 1)
        ax2.plot(se, smooth(g_losses), label='G (smoothed)', color='#E74C3C', linewidth=2)
        ax2.plot(se, smooth(d_losses), label='D (smoothed)', color='#3498DB', linewidth=2)
        ax2.set_title(f"{title} – Smoothed (window={w})")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
        ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(filename, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()


def real_labels(size):
    return torch.FloatTensor(size, 1).uniform_(0.85, 1.0)

def fake_labels(size):
    return torch.FloatTensor(size, 1).uniform_(0.0, 0.15)


van_G_losses, van_D_losses = [], []
c_G_losses,   c_D_losses   = [], []

print("=" * 60)
print("Starting training (CPU mode) …")
print("=" * 60)

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    van_G.train(); van_D.train()
    c_G.train();   c_D.train()

    ep_van_G, ep_van_D = [], []
    ep_c_G,   ep_c_D   = [], []

    for real_imgs, labels in train_loader:
        B          = real_imgs.size(0)
        real_flat  = real_imgs.view(B, -1)     

        real_tgt = real_labels(B)
        fake_tgt = fake_labels(B)

     
        opt_van_D.zero_grad()
        z         = torch.randn(B, LATENT_DIM)
        fake_flat = van_G(z).detach()
        loss_vD   = (criterion(van_D(real_flat), real_tgt) +
                     criterion(van_D(fake_flat), fake_tgt))
        loss_vD.backward()
        opt_van_D.step()

 
        opt_van_G.zero_grad()
        z       = torch.randn(B, LATENT_DIM)
        loss_vG = criterion(van_D(van_G(z)), real_labels(B))
        loss_vG.backward()
        opt_van_G.step()

        ep_van_G.append(loss_vG.item())
        ep_van_D.append(loss_vD.item())

       
        opt_c_D.zero_grad()
        z         = torch.randn(B, LATENT_DIM)
        fake_flat = c_G(z, labels).detach()
        loss_cD   = (criterion(c_D(real_flat, labels), real_tgt) +
                     criterion(c_D(fake_flat, labels), fake_tgt))
        loss_cD.backward()
        opt_c_D.step()

   
        opt_c_G.zero_grad()
        z       = torch.randn(B, LATENT_DIM)
        loss_cG = criterion(c_D(c_G(z, labels), labels), real_labels(B))
        loss_cG.backward()
        opt_c_G.step()

        ep_c_G.append(loss_cG.item())
        ep_c_D.append(loss_cD.item())

    van_G_losses.append(np.mean(ep_van_G))
    van_D_losses.append(np.mean(ep_van_D))
    c_G_losses.append(np.mean(ep_c_G))
    c_D_losses.append(np.mean(ep_c_D))

    elapsed = (time.time() - start_time) / 60
    print(f"Epoch [{epoch:3d}/{NUM_EPOCHS}] | "
          f"VanG: {van_G_losses[-1]:.4f}  VanD: {van_D_losses[-1]:.4f} | "
          f"cG: {c_G_losses[-1]:.4f}  cD: {c_D_losses[-1]:.4f} | "
          f"{elapsed:.1f} min")

    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        van_G.eval(); c_G.eval()
        with torch.no_grad():
            van_out = van_G(fixed_noise).view(-1, CHANNELS, IMG_SIZE, IMG_SIZE)
            c_out   = c_G(fixed_noise, fixed_labels).view(-1, CHANNELS, IMG_SIZE, IMG_SIZE)
        save_image_grid(van_out, epoch, "vanilla_gan", title="Vanilla GAN")
        save_image_grid(c_out,  epoch, "cgan",         title="cGAN")
        van_G.train(); c_G.train()

print(f"\n Training complete in {(time.time()-start_time)/60:.1f} min")


plot_losses(van_G_losses, van_D_losses,
            title="Vanilla GAN", filename="vanilla_gan_losses.png")

plot_losses(c_G_losses, c_D_losses,
            title="Conditional GAN (cGAN)", filename="cgan_losses.png")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("GAN vs cGAN – Loss Comparison", fontsize=15, fontweight='bold')
ep = range(1, NUM_EPOCHS + 1)

axes[0,0].plot(ep, van_G_losses, color='#E74C3C', label='Vanilla GAN', lw=1.5)
axes[0,0].plot(ep, c_G_losses,   color='#9B59B6', label='cGAN',        lw=1.5)
axes[0,0].set_title("Generator Loss"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ep, van_D_losses, color='#3498DB', label='Vanilla GAN', lw=1.5)
axes[0,1].plot(ep, c_D_losses,   color='#1ABC9C', label='cGAN',        lw=1.5)
axes[0,1].set_title("Discriminator Loss"); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

van_tot = [g+d for g,d in zip(van_G_losses, van_D_losses)]
c_tot   = [g+d for g,d in zip(c_G_losses,   c_D_losses)]
axes[1,0].plot(ep, van_tot, color='#E67E22', label='Vanilla GAN', lw=1.5)
axes[1,0].plot(ep, c_tot,   color='#2ECC71', label='cGAN',        lw=1.5)
axes[1,0].set_title("Total Loss (G+D)"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(ep, [abs(d-1.0) for d in van_D_losses], color='#95A5A6', label='Vanilla GAN', lw=1.5)
axes[1,1].plot(ep, [abs(d-1.0) for d in c_D_losses],   color='#F39C12', label='cGAN',        lw=1.5)
axes[1,1].set_title("|D_loss − 1.0| (Equilibrium)"); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("gan_vs_cgan_comparison.png", dpi=120, bbox_inches='tight')
plt.show()
print("Saved: gan_vs_cgan_comparison.png")

def generate_per_class_grid(model, is_conditional=True, n_samples=SAMPLES_PER_CLASS):
    model.eval()
    fig = plt.figure(figsize=(16, 18))
    gs  = gridspec.GridSpec(NUM_CLASSES, n_samples + 1,
                            width_ratios=[0.6] + [1]*n_samples,
                            hspace=0.1, wspace=0.05)
    name = "cGAN" if is_conditional else "Vanilla GAN"
    fig.suptitle(f"{name} – {n_samples} Samples per Class",
                 fontsize=16, fontweight='bold', y=1.01)

    with torch.no_grad():
        for cls in range(NUM_CLASSES):
            z      = torch.randn(n_samples, LATENT_DIM)
            labels = torch.full((n_samples,), cls, dtype=torch.long)
            imgs   = model(z, labels) if is_conditional else model(z)
            imgs   = imgs.view(n_samples, IMG_SIZE, IMG_SIZE).numpy()
            imgs   = (imgs + 1) / 2

            ax_lbl = fig.add_subplot(gs[cls, 0])
            ax_lbl.text(0.5, 0.5, CLASS_NAMES[cls],
                        va='center', ha='center', fontsize=9, fontweight='bold',
                        transform=ax_lbl.transAxes)
            ax_lbl.axis('off')

            for s in range(n_samples):
                ax = fig.add_subplot(gs[cls, s + 1])
                ax.imshow(imgs[s], cmap='gray', vmin=0, vmax=1)
                ax.axis('off')

    fname = f"{'cgan' if is_conditional else 'vanilla'}_per_class_grid.png"
    plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {fname}")

generate_per_class_grid(c_G,   is_conditional=True)
generate_per_class_grid(van_G, is_conditional=False)

van_G.eval(); c_G.eval()
fig, axes = plt.subplots(NUM_CLASSES, 2*SAMPLES_PER_CLASS + 1, figsize=(22, 18))
fig.suptitle("Vanilla GAN  ↔  cGAN  (Left = Vanilla, Right = cGAN)",
             fontsize=13, fontweight='bold')

with torch.no_grad():
    for cls in range(NUM_CLASSES):
        z      = torch.randn(SAMPLES_PER_CLASS, LATENT_DIM)
        labels = torch.full((SAMPLES_PER_CLASS,), cls, dtype=torch.long)
        vi = ((van_G(z).view(SAMPLES_PER_CLASS, IMG_SIZE, IMG_SIZE).numpy() + 1) / 2)
        ci = ((c_G(z, labels).view(SAMPLES_PER_CLASS, IMG_SIZE, IMG_SIZE).numpy() + 1) / 2)

        for s in range(SAMPLES_PER_CLASS):
            axes[cls, s].imshow(vi[s], cmap='gray', vmin=0, vmax=1); axes[cls, s].axis('off')

        axes[cls, SAMPLES_PER_CLASS].axis('off')
        axes[cls, SAMPLES_PER_CLASS].text(
            0.5, 0.5, CLASS_NAMES[cls], va='center', ha='center',
            fontsize=7.5, fontweight='bold', transform=axes[cls, SAMPLES_PER_CLASS].transAxes,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.8))

        for s in range(SAMPLES_PER_CLASS):
            axes[cls, SAMPLES_PER_CLASS+1+s].imshow(ci[s], cmap='gray', vmin=0, vmax=1)
            axes[cls, SAMPLES_PER_CLASS+1+s].axis('off')

fig.text(0.26, 1.00, "← Vanilla GAN →", ha='center', fontsize=11, color='#E74C3C', fontweight='bold')
fig.text(0.74, 1.00, "←    cGAN    →",  ha='center', fontsize=11, color='#9B59B6', fontweight='bold')
plt.tight_layout()
plt.savefig("gan_vs_cgan_sidebyside.png", dpi=130, bbox_inches='tight')
plt.show()
plt.close()
print("Saved: gan_vs_cgan_sidebyside.png")

print("=" * 65)
print("               EVALUATION SUMMARY")
print("=" * 65)
print(f"\n{'Metric':<30} {'Vanilla GAN':>12}  {'cGAN':>10}")
print("-" * 56)
print(f"{'Final G Loss':<30} {van_G_losses[-1]:>12.4f}  {c_G_losses[-1]:>10.4f}")
print(f"{'Final D Loss':<30} {van_D_losses[-1]:>12.4f}  {c_D_losses[-1]:>10.4f}")
print(f"{'G-Loss Std (last 10 ep)':<30} {np.std(van_G_losses[-10:]):>12.4f}  {np.std(c_G_losses[-10:]):>10.4f}")
print(f"{'D-Loss Std (last 10 ep)':<30} {np.std(van_D_losses[-10:]):>12.4f}  {np.std(c_D_losses[-10:]):>10.4f}")

print("\n cGAN enables class-specific generation via label embeddings.")
print(" Vanilla GAN output is diverse but class-uncontrolled.")
print("  For GPU training: switch to a Kaggle kernel with T4/A100")
print("   (sm_70+) and set device='cuda' at the top of the notebook.")


torch.save({'epoch': NUM_EPOCHS, 'G': van_G.state_dict(), 'D': van_D.state_dict(),
            'g_losses': van_G_losses, 'd_losses': van_D_losses}, 'vanilla_gan_ckpt.pth')
torch.save({'epoch': NUM_EPOCHS, 'G': c_G.state_dict(),   'D': c_D.state_dict(),
            'g_losses': c_G_losses,   'd_losses': c_D_losses},   'cgan_ckpt.pth')
print("\n Checkpoints saved.")


def generate_class_samples(class_id, n=10):
    c_G.eval()
    with torch.no_grad():
        z      = torch.randn(n, LATENT_DIM)
        labels = torch.full((n,), class_id, dtype=torch.long)
        imgs   = ((c_G(z, labels).view(n, IMG_SIZE, IMG_SIZE).numpy() + 1) / 2)
    fig, axes = plt.subplots(1, n, figsize=(n*1.4, 1.8))
    fig.suptitle(f'cGAN → {CLASS_NAMES[class_id]}', fontsize=11, fontweight='bold')
    for i, ax in enumerate(axes):
        ax.imshow(imgs[i], cmap='gray', vmin=0, vmax=1); ax.axis('off')
    plt.tight_layout()
    plt.savefig(f"inference_class_{class_id}.png", dpi=110, bbox_inches='tight')
    plt.show()

generate_class_samples(7)   
generate_class_samples(8)  

Running on : cpu
Note: CUDA P100 (sm_60) is incompatible with this PyTorch build → CPU mode.
Batch size : 256
Epochs     : 50
Device     : cpu


100%|██████████| 26.4M/26.4M [00:01<00:00, 18.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 305kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.59MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.7MB/s]


Dataset size : 60,000
Batches/epoch: 235
Saved: sample_images.png
Vanilla G  params: 1,489,936
Vanilla D  params: 533,505
cGAN    G  params: 1,503,236
cGAN    D  params: 559,605
Starting training (CPU mode) …
Epoch [  1/50] | VanG: 1.9544  VanD: 0.6948 | cG: 2.0649  cD: 0.6770 | 0.7 min
Epoch [  2/50] | VanG: 1.0002  VanD: 1.2454 | cG: 1.3893  cD: 1.1328 | 1.4 min
Epoch [  3/50] | VanG: 0.8389  VanD: 1.2654 | cG: 1.0469  cD: 1.2061 | 2.0 min
Epoch [  4/50] | VanG: 0.8656  VanD: 1.2634 | cG: 0.9931  cD: 1.2118 | 2.7 min
Epoch [  5/50] | VanG: 0.8622  VanD: 1.2647 | cG: 0.9909  cD: 1.2201 | 3.3 min
Epoch [  6/50] | VanG: 0.8324  VanD: 1.2859 | cG: 0.9217  cD: 1.2661 | 4.0 min
Epoch [  7/50] | VanG: 0.8334  VanD: 1.2890 | cG: 0.8720  cD: 1.3086 | 4.7 min
Epoch [  8/50] | VanG: 0.8298  VanD: 1.2989 | cG: 0.8148  cD: 1.3347 | 5.4 min
Epoch [  9/50] | VanG: 0.8242  VanD: 1.3073 | cG: 0.8194  cD: 1.3338 | 6.1 min
Epoch [ 10/50] | VanG: 0.8136  VanD: 1.3192 | cG: 0.8026  cD: 1.3451 | 6.8 min
E